# Network Authentication Classification Analysis
## Following CRISP-DM Methodology and ML Reference Guide

---

## Project Overview

**Problem Statement**: Because authentication failures are rare and context-dependent, we reframed the problem from predicting failures to modeling normal authentication behavior. Using CRISP-DM, we first conducted exploratory and information-theoretic analysis to identify which features encode behavior versus identity. This allowed us to design aggregated, window-based features that align with enterprise authentication processes, enabling robust anomaly detection rather than brittle classification.

---

## Refined Problem Statement

### Research Question
**Can we effectively distinguish between normal and anomalous enterprise authentication patterns by modeling behavioral features (authentication type, logon type, activity orientation) rather than identity features, and what classification techniques best capture these patterns given severe class imbalance (~1.2% failure rate)?**

### Data Source
- **Dataset**: Los Alamos National Laboratory (LANL) Unified Host and Network Dataset
- **File**: `auth.txt` (~73GB, 1+ billion authentication events)
- **Sample**: 500,000 stratified random samples using reservoir sampling
- **Structure**: 9 columns (Time*, SourceUser, DestUser, SourceComputer, DestComputer, AuthType, LogonType, Activity, Status)
- **Note**: *Time column is anonymized/classified - temporal analysis not possible

### Expected Techniques
1. **Feature Engineering**: OneHotEncoding for low-cardinality behavioral features (AuthType, LogonType, Activity)
2. **Identity-based Aggregation**: Behavioral profiles per user/computer (alternative to temporal windows)
3. **Dimensionality Reduction**: PCA to understand feature relationships
4. **Classification Models**: Logistic Regression, Decision Trees, KNN, SVM (with class weight balancing)
5. **Evaluation**: Precision, Recall, F1-score (focus on minority class performance)

### Expected Results
- Identification of behavioral patterns that distinguish successful vs failed authentications
- Model performance comparison with standardized reporting
- Feature importance ranking to understand authentication failure predictors
- Anomaly detection baseline for enterprise security monitoring

### Why This Question is Important
Enterprise authentication logs represent a critical security monitoring surface. With billions of daily authentication events, manual review is impossible. A robust classification/anomaly detection approach enables:
- Proactive identification of credential compromise attempts
- Reduced false positive rates through behavioral modeling
- Scalable security monitoring for large organizations

---

## Table of Contents
1. [Setup and Imports](#1-setup-and-imports)
2. [Data Loading and Sampling](#2-data-loading)
3. [Data Cleaning](#3-data-cleaning) — missing values, duplicates, cardinality
4. [Exploratory Data Analysis (EDA)](#4-eda) — visualizations and distributions
5. [Feature Engineering](#5-feature-engineering) — behavioral encoding, user profiles, outlier analysis
6. [Model Building and Comparison](#6-model-building)
7. [Model Evaluation and Visualization](#7-model-reporting) — confusion matrices, metric comparison
8. [Conclusions and Next Steps](#8-conclusions)

In [4]:
## Chunked Approximate Reservoir (batch-wise)
## statistically faithful approximation of reservoir sampling that operates per chunk instead of per row
## Operates on vectorized chunks, Uses pandas’ optimized .sample(), Reduces Python loop overhead by orders of magnitude

In [5]:
import pandas as pd
import numpy as np

column_names = [
    'Time', 'SourceUser', 'DestUser',
    'SourceComputer', 'DestComputer',
    'AuthType', 'LogonType', 'Activity', 'Status'
]

SAMPLE_SIZE = 500_000
CHUNKSIZE = 1_000_000
RANDOM_STATE = 42

reservoir = None
rows_seen = 0

for chunk in pd.read_csv(
    "auth.txt",
    names=column_names,
    header=None,
    chunksize=CHUNKSIZE,
    low_memory=False
):
    rows_seen += len(chunk)

    if reservoir is None:
        reservoir = chunk.sample(
            n=min(SAMPLE_SIZE, len(chunk)),
            random_state=RANDOM_STATE
        )
        continue

    p = SAMPLE_SIZE / rows_seen
    take = chunk.sample(frac=p, random_state=RANDOM_STATE)

    reservoir = pd.concat([reservoir, take], ignore_index=True)

    if len(reservoir) > SAMPLE_SIZE:
        reservoir = reservoir.sample(
            n=SAMPLE_SIZE,
            random_state=RANDOM_STATE
        )

df = reservoir.reset_index(drop=True)

print(df.shape)


(500000, 9)


In [6]:
df['Status'].value_counts(normalize=True)

Status
Success    0.98752
Fail       0.01248
Name: proportion, dtype: float64

---
## 3. Data Cleaning

**Reference**: ML_Reference_Guide - Section 3

Before modeling, we must verify data quality:
1. **Data types** — confirm each column has the expected type
2. **Missing values** — identify and handle nulls
3. **Duplicates** — detect and remove exact duplicate rows
4. **Basic statistics** — understand the shape and structure of the sample

In [ ]:
# =============================================================================
# 3.1 DATA TYPES AND STRUCTURE
# =============================================================================

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
print("Column data types:")
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print("\nFirst 5 rows:")
df.head()

In [ ]:
# =============================================================================
# 3.2 MISSING VALUES ANALYSIS
# =============================================================================

missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
print("Missing Values Summary:")
print(missing_summary)

# Check for placeholder missing values (e.g., '?' in this dataset)
print("\nPlaceholder '?' counts per column:")
for col in df.columns:
    q_count = (df[col] == '?').sum()
    if q_count > 0:
        print(f"  {col}: {q_count:,} ({q_count/len(df)*100:.2f}%)")

In [ ]:
# =============================================================================
# 3.3 DUPLICATE ANALYSIS
# =============================================================================

dup_count = df.duplicated().sum()
print(f"Exact duplicate rows: {dup_count:,} ({dup_count/len(df)*100:.2f}%)")

if dup_count > 0:
    print(f"\nRemoving {dup_count:,} duplicates...")
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Dataset after dedup: {df.shape[0]:,} rows × {df.shape[1]} columns")
else:
    print("No duplicates found — dataset is clean.")

In [ ]:
# =============================================================================
# 3.4 UNIQUE VALUE COUNTS PER COLUMN (Cardinality Overview)
# =============================================================================

print("Unique values per column:")
print("-" * 45)
for col in df.columns:
    n_unique = df[col].nunique()
    print(f"  {col:<20s} {n_unique:>8,} unique")
print("-" * 45)
print(f"  {'Total rows':<20s} {len(df):>8,}")

---
## 4. Exploratory Data Analysis (EDA) — Visualizations

Visual exploration of feature distributions and class imbalance to guide modeling decisions.

In [ ]:
# =============================================================================
# 4.1 CLASS IMBALANCE VISUALIZATION
# =============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart of class distribution
status_counts = df['Status'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(status_counts.index, status_counts.values, color=colors, edgecolor='black')
axes[0].set_title('Authentication Status Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Status', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
for i, (label, val) in enumerate(zip(status_counts.index, status_counts.values)):
    axes[0].text(i, val + 1000, f'{val:,}', ha='center', fontsize=11, fontweight='bold')

# Pie chart showing the imbalance
axes[1].pie(status_counts.values, labels=status_counts.index, autopct='%1.2f%%',
            colors=colors, startangle=90, textprops={'fontsize': 12},
            explode=(0, 0.1), shadow=True)
axes[1].set_title('Class Imbalance — Fail is ~1.2%', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nImbalance ratio: {status_counts['Success'] / status_counts['Fail']:.0f}:1 (Success:Fail)")

In [ ]:
# =============================================================================
# 4.2 CATEGORICAL FEATURE DISTRIBUTIONS
# Behavioral features: AuthType, LogonType, Activity
# =============================================================================

categorical_cols = ['AuthType', 'LogonType', 'Activity']

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, col in enumerate(categorical_cols):
    counts = df[col].value_counts()
    # Truncate long AuthType labels for readability
    labels = [str(x)[:25] + '...' if len(str(x)) > 25 else str(x) for x in counts.index]
    
    axes[i].barh(labels, counts.values, color=sns.color_palette("viridis", len(counts)))
    axes[i].set_title(f'{col} Distribution', fontsize=14, fontweight='bold')
    axes[i].set_xlabel('Count', fontsize=12)
    axes[i].invert_yaxis()
    
    # Add count labels on bars
    for j, val in enumerate(counts.values):
        axes[i].text(val + max(counts.values)*0.01, j, f'{val:,}', 
                     va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 4.3 FEATURE DISTRIBUTIONS BY STATUS (Success vs Fail)
# Shows which behavioral feature values are associated with failures
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for i, col in enumerate(categorical_cols):
    # Calculate proportions within each category
    ct = pd.crosstab(df[col], df['Status'], normalize='index') * 100
    ct = ct.sort_values('Fail', ascending=True)
    
    # Truncate labels
    labels = [str(x)[:25] + '...' if len(str(x)) > 25 else str(x) for x in ct.index]
    
    ct.plot(kind='barh', stacked=True, ax=axes[i], color=['#2ecc71', '#e74c3c'],
            edgecolor='black', linewidth=0.5)
    axes[i].set_title(f'Failure Rate by {col}', fontsize=14, fontweight='bold')
    axes[i].set_xlabel('Percentage (%)', fontsize=12)
    axes[i].set_yticklabels(labels, fontsize=10)
    axes[i].legend(title='Status', fontsize=10)
    axes[i].set_xlim(0, 105)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 4.4 TOP SOURCE AND DESTINATION COMPUTERS (Identity Feature Overview)
# Shows high-cardinality identity features are not suitable for OHE
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, col in enumerate(['SourceComputer', 'DestComputer']):
    top15 = df[col].value_counts().head(15)
    axes[i].barh(top15.index[::-1], top15.values[::-1], 
                 color=sns.color_palette("coolwarm", 15))
    axes[i].set_title(f'Top 15 {col} by Event Count', fontsize=14, fontweight='bold')
    axes[i].set_xlabel('Event Count', fontsize=12)
    for j, val in enumerate(top15.values[::-1]):
        axes[i].text(val + max(top15.values)*0.01, j, f'{val:,}', 
                     va='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nSourceComputer unique values: {df['SourceComputer'].nunique():,}")
print(f"DestComputer unique values:   {df['DestComputer'].nunique():,}")
print("→ Too high cardinality for OneHotEncoding; use aggregation instead.")

In [ ]:
# =============================================================================
# 4.5 CORRELATION HEATMAP OF BEHAVIORAL FEATURES
# Encoded categorical features — checks for multicollinearity
# =============================================================================

from sklearn.preprocessing import OneHotEncoder

# Temporarily encode for correlation analysis
_ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
_X_temp = _ohe.fit_transform(df[['AuthType', 'LogonType', 'Activity']])
_feature_names = _ohe.get_feature_names_out(['AuthType', 'LogonType', 'Activity'])

# Shorten names for readability
short_names = [n.replace('AuthType_', 'Auth:').replace('LogonType_', 'Logon:')
               .replace('Activity_', 'Act:')[:20] for n in _feature_names]

corr_matrix = pd.DataFrame(_X_temp, columns=short_names).corr()

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0,
            annot=False, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8, "label": "Correlation"},
            ax=ax)
ax.set_title('Correlation Heatmap — Encoded Behavioral Features', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Clean up temporary variables
del _ohe, _X_temp, _feature_names

In [7]:
y = df["Status"]
y.value_counts(normalize=True)

Status
Success    0.98752
Fail       0.01248
Name: proportion, dtype: float64

---
## 5. Feature Engineering

**Reference**: ML_Reference_Guide - Section 5

### Feature Categories

Based on our entropy and mutual information analysis, we can categorize features:

| Category | Features | Characteristics | Encoding Strategy |
|----------|----------|-----------------|-------------------|
| **Identity** | SourceUser, DestUser, SourceComputer, DestComputer | High cardinality (~10k-30k unique) | Aggregation, not OHE |
| **Behavioral** | AuthType, LogonType, Activity | Low cardinality (7-31 unique) | OneHotEncoding |
| **Outcome** | Status | Binary target (Success/Fail) | Label Encoding |
| **Temporal** | Time | **ANONYMIZED** - Not usable for temporal analysis | Cannot use |

### Data Limitation: Anonymized Timestamps

**Important**: The Time column in this LANL dataset has been anonymized/classified by the laboratory for security reasons. The values do not represent actual timestamps and cannot be used for:
- Temporal window-based aggregation
- Time-of-day patterns
- Sequence analysis

**Alternative Approach**: Instead of temporal windows, we aggregate behavioral features by identity (user/computer) to capture behavioral patterns.

### Why This Matters for Classification

- **Identity features** have too high cardinality for OneHotEncoding - would create 30k+ sparse columns
- **Behavioral features** encode *how* authentication happens (protocol, type, action)
- **High mutual information** between AuthType-Activity (0.69) and AuthType-LogonType (0.44) suggests redundancy
- **Low mutual information** with Status indicates we need engineered features

In [8]:
# =============================================================================
# 5.1 BEHAVIORAL FEATURE ENCODING
# Reference: ML_Reference_Guide - Section 5.3
# =============================================================================

from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# Define feature categories
behavioral_cols = ['AuthType', 'LogonType', 'Activity']
identity_cols = ['SourceUser', 'DestUser', 'SourceComputer', 'DestComputer']
target_col = 'Status'

# Create OneHotEncoder for behavioral features
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_behavioral = ohe.fit_transform(df[behavioral_cols])

# Get feature names for interpretability
behavioral_feature_names = ohe.get_feature_names_out(behavioral_cols)
print(f"Created {len(behavioral_feature_names)} behavioral features:")
print(behavioral_feature_names)

Created 31 behavioral features:
['AuthType_?' 'AuthType_Kerberos' 'AuthType_MICROSOFT_AUTHENTICATION_PA'
 'AuthType_MICROSOFT_AUTHENTICATION_PAC'
 'AuthType_MICROSOFT_AUTHENTICATION_PACK'
 'AuthType_MICROSOFT_AUTHENTICATION_PACKA'
 'AuthType_MICROSOFT_AUTHENTICATION_PACKAG'
 'AuthType_MICROSOFT_AUTHENTICATION_PACKAGE'
 'AuthType_MICROSOFT_AUTHENTICATION_PACKAGE_'
 'AuthType_MICROSOFT_AUTHENTICATION_PACKAGE_V1'
 'AuthType_MICROSOFT_AUTHENTICATION_PACKAGE_V1_'
 'AuthType_MICROSOFT_AUTHENTICATION_PACKAGE_V1_0' 'AuthType_NTLM'
 'AuthType_Negotiate' 'LogonType_?' 'LogonType_Batch'
 'LogonType_CachedInteractive' 'LogonType_Interactive' 'LogonType_Network'
 'LogonType_NetworkCleartext' 'LogonType_NewCredentials'
 'LogonType_RemoteInteractive' 'LogonType_Service' 'LogonType_Unlock'
 'Activity_AuthMap' 'Activity_LogOff' 'Activity_LogOn'
 'Activity_ScreenLock' 'Activity_ScreenUnlock' 'Activity_TGS'
 'Activity_TGT']


In [3]:
# Display first 5 behavioral feature names (numpy array, not DataFrame)
behavioral_feature_names[:5]


NameError: name 'behavioral_feature_names' is not defined

In [13]:
# =============================================================================
# 5.2 IDENTITY-BASED BEHAVIORAL AGGREGATION (Alternative to Temporal Windows)
# Reference: ML_Reference_Guide - Section 5.1 (Creating New Features)
# =============================================================================

# NOTE: The Time column in this dataset has been anonymized/classified by the laboratory,
# so temporal window-based features are not meaningful. Instead, we aggregate by
# identity (user/computer) to capture behavioral patterns.

def create_user_behavioral_profile(df, user_col='SourceUser'):
    """
    Create aggregated behavioral features per user (without temporal component).
    This captures user-level behavioral patterns.
    
    Parameters:
    -----------
    df : DataFrame with authentication events
    user_col : Column to group by (user or computer)
    
    Returns:
    --------
    DataFrame with user-level aggregated features
    """
    # Group by user
    user_aggs = df.groupby(user_col).agg({
        'Time': 'count',  # Total event count per user
        'SourceComputer': 'nunique',  # Unique source computers used
        'DestComputer': 'nunique',  # Unique destination computers accessed
        'AuthType': lambda x: x.value_counts().index[0] if len(x) > 0 else 'Unknown',  # Primary auth type
        'Activity': lambda x: (x == 'LogOn').sum() / len(x) if len(x) > 0 else 0,  # LogOn ratio
        'Status': lambda x: (x == 'Fail').sum() / len(x) if len(x) > 0 else 0,  # Failure rate
    }).reset_index()
    
    user_aggs.columns = [user_col, 'event_count', 'unique_src_computers', 
                         'unique_dst_computers', 'primary_auth_type', 'logon_ratio', 'failure_rate']
    
    return user_aggs

# Calculate user behavioral profiles
print("Creating user behavioral profiles (non-temporal aggregation)...")
user_profiles = create_user_behavioral_profile(df, user_col='SourceUser')
print(f"Created {len(user_profiles)} user profiles")
print(f"\nSample profiles:")
print(user_profiles.head(10))
print(f"\nUsers with highest failure rates:")
print(user_profiles.nlargest(10, 'failure_rate')[['SourceUser', 'event_count', 'failure_rate']])

Creating user behavioral profiles (non-temporal aggregation)...
Created 26863 user profiles

Sample profiles:
               SourceUser  event_count  unique_src_computers  \
0     ANONYMOUS LOGON@C10            2                     2   
1  ANONYMOUS LOGON@C10049            1                     1   
2  ANONYMOUS LOGON@C10060            1                     1   
3  ANONYMOUS LOGON@C10071            1                     1   
4    ANONYMOUS LOGON@C101            1                     1   
5   ANONYMOUS LOGON@C1012            1                     1   
6  ANONYMOUS LOGON@C10171            5                     1   
7  ANONYMOUS LOGON@C10239            1                     1   
8  ANONYMOUS LOGON@C10271            1                     1   
9  ANONYMOUS LOGON@C10311            1                     1   

   unique_dst_computers primary_auth_type  logon_ratio  failure_rate  
0                     1              NTLM          1.0           0.0  
1                     1              NTLM  

---
### 5.3 Outlier Analysis — User Behavioral Profiles

Outlier detection on the aggregated user profiles helps identify unusual accounts
(e.g., service accounts with extreme event counts, or users with anomalously high failure rates).
These are important for security monitoring and may also affect model performance.

In [ ]:
# =============================================================================
# 5.3 OUTLIER ANALYSIS ON USER BEHAVIORAL PROFILES
# =============================================================================

numeric_profile_cols = ['event_count', 'unique_src_computers', 'unique_dst_computers',
                        'logon_ratio', 'failure_rate']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_profile_cols):
    axes[i].boxplot(user_profiles[col], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#3498db', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(f'{col}', fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Value', fontsize=11)
    
    # Annotate IQR-based outlier count
    Q1 = user_profiles[col].quantile(0.25)
    Q3 = user_profiles[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((user_profiles[col] < lower) | (user_profiles[col] > upper)).sum()
    axes[i].text(0.95, 0.95, f'Outliers: {n_outliers:,}',
                 transform=axes[i].transAxes, ha='right', va='top',
                 fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Hide unused subplot
axes[5].axis('off')

plt.suptitle('Outlier Analysis — User Behavioral Profiles (IQR Method)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Summary statistics
print("\nUser Profile Summary Statistics:")
print(user_profiles[numeric_profile_cols].describe().round(4).to_string())

In [ ]:
# =============================================================================
# 5.4 FAILURE RATE DISTRIBUTION (Log-scale histogram)
# Identifies users with anomalously high failure rates
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Event count distribution (log scale)
axes[0].hist(user_profiles['event_count'], bins=50, color='#3498db', 
             edgecolor='black', alpha=0.8)
axes[0].set_yscale('log')
axes[0].set_title('User Event Count Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Events per User', fontsize=12)
axes[0].set_ylabel('Number of Users (log scale)', fontsize=12)

# Failure rate distribution (excluding zero-failure users for clarity)
fail_users = user_profiles[user_profiles['failure_rate'] > 0]
axes[1].hist(fail_users['failure_rate'], bins=30, color='#e74c3c',
             edgecolor='black', alpha=0.8)
axes[1].set_title(f'Failure Rate Distribution (Users with Failures, n={len(fail_users):,})',
                  fontsize=14, fontweight='bold')
axes[1].set_xlabel('Failure Rate', fontsize=12)
axes[1].set_ylabel('Number of Users', fontsize=12)

plt.tight_layout()
plt.show()

# Key outlier stats
high_fail = user_profiles[user_profiles['failure_rate'] == 1.0]
print(f"\nUsers with 100% failure rate: {len(high_fail):,}")
print(f"Users with any failures:     {len(fail_users):,} / {len(user_profiles):,} "
      f"({len(fail_users)/len(user_profiles)*100:.1f}%)")

---
## 6. Model Building and Comparison

**Reference**: ML_Reference_Guide - Sections 6, 8, 14, 16

### Classification Models for Imbalanced Data

Given the severe class imbalance (~98.8% Success, ~1.2% Fail), we need to consider:

| Model | Handles Imbalance | Training Speed | Interpretability | Notes |
|-------|-------------------|----------------|------------------|-------|
| Logistic Regression | Yes (class_weight) | Fast | High | Good for baseline, interpretable coefficients |
| Decision Tree | Yes (class_weight) | Fast | Very High | Visual rules, prone to overfit |
| KNN | Limited | Fast (train), Slow (predict) | Low | Distance-based, sensitive to scale |
| SVM | Yes (class_weight) | Medium-Slow | Low (kernel) | Good margins, expensive for large N |

### Strategy
1. Use `class_weight='balanced'` to handle imbalance
2. Focus on **Precision** and **Recall** for the minority class (Fail)
3. Compare training time and interpretability trade-offs

In [14]:
# =============================================================================
# 7. STANDARDIZED MODEL REPORTING FUNCTION
# Reference: ML_Reference_Guide - Section 8 (Model Evaluation)
# =============================================================================

import time
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate_classifier(model, X_train, X_test, y_train, y_test, model_name="Model"):
    """
    Standardized classifier evaluation and reporting function.
    
    Parameters:
    -----------
    model : sklearn classifier (fitted or unfitted)
    X_train, X_test : Feature matrices
    y_train, y_test : Target vectors
    model_name : String name for display
    
    Returns:
    --------
    Dictionary with all metrics
    """
    results = {'model_name': model_name}
    
    # Training time
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    results['train_time'] = train_time
    
    # Predictions
    start_time = time.time()
    y_pred = model.predict(X_test)
    predict_time = time.time() - start_time
    results['predict_time'] = predict_time
    
    # Overall metrics
    results['accuracy'] = accuracy_score(y_test, y_pred)
    
    # Per-class metrics (focus on minority class)
    results['precision_fail'] = precision_score(y_test, y_pred, pos_label='Fail', zero_division=0)
    results['recall_fail'] = recall_score(y_test, y_pred, pos_label='Fail', zero_division=0)
    results['f1_fail'] = f1_score(y_test, y_pred, pos_label='Fail', zero_division=0)
    
    results['precision_success'] = precision_score(y_test, y_pred, pos_label='Success', zero_division=0)
    results['recall_success'] = recall_score(y_test, y_pred, pos_label='Success', zero_division=0)
    results['f1_success'] = f1_score(y_test, y_pred, pos_label='Success', zero_division=0)
    
    # Macro average (treats both classes equally)
    results['f1_macro'] = f1_score(y_test, y_pred, average='macro', zero_division=0)
    
    return results, y_pred

def print_classification_report(results, y_test, y_pred):
    """
    Print a standardized classification report.
    """
    print("=" * 70)
    print(f"MODEL: {results['model_name']}")
    print("=" * 70)
    print(f"\nTiming:")
    print(f"  Training Time:   {results['train_time']:.4f} seconds")
    print(f"  Prediction Time: {results['predict_time']:.4f} seconds")
    
    print(f"\nOverall Performance:")
    print(f"  Accuracy:        {results['accuracy']:.4f}")
    print(f"  Macro F1:        {results['f1_macro']:.4f}")
    
    print(f"\nMinority Class (Fail) Performance:")
    print(f"  Precision:       {results['precision_fail']:.4f}")
    print(f"  Recall:          {results['recall_fail']:.4f}")
    print(f"  F1-Score:        {results['f1_fail']:.4f}")
    
    print(f"\nMajority Class (Success) Performance:")
    print(f"  Precision:       {results['precision_success']:.4f}")
    print(f"  Recall:          {results['recall_success']:.4f}")
    print(f"  F1-Score:        {results['f1_success']:.4f}")
    
    print(f"\nConfusion Matrix:")
    cm = confusion_matrix(y_test, y_pred, labels=['Success', 'Fail'])
    print(f"              Predicted")
    print(f"              Success  Fail")
    print(f"  Actual Success  {cm[0,0]:6d}  {cm[0,1]:6d}")
    print(f"  Actual Fail     {cm[1,0]:6d}  {cm[1,1]:6d}")
    print("-" * 70)
    
def plot_confusion_matrix(y_test, y_pred, model_name):
    """Plot confusion matrix heatmap."""
    cm = confusion_matrix(y_test, y_pred, labels=['Success', 'Fail'])
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Success', 'Fail'],
                yticklabels=['Success', 'Fail'])
    plt.title(f'Confusion Matrix: {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

print("Standardized reporting functions defined successfully!")

Standardized reporting functions defined successfully!


In [15]:
# =============================================================================
# 6.1 PREPARE DATA FOR CLASSIFICATION
# Reference: ML_Reference_Guide - Section 6 (Data Splitting)
# =============================================================================

from sklearn.model_selection import train_test_split

# Create feature matrix from behavioral features only
X = pd.DataFrame(X_behavioral, columns=behavioral_feature_names)

# Target variable
y = df[target_col]

# Train-test split (stratified to maintain class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3, 
    random_state=42,
    stratify=y  # Maintain class proportions
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nClass distribution in training set:")
print(y_train.value_counts())
print(f"\nClass distribution in test set:")
print(y_test.value_counts())
print(f"\nFailure rate (train): {(y_train == 'Fail').mean():.2%}")
print(f"Failure rate (test):  {(y_test == 'Fail').mean():.2%}")

Training set: 350000 samples
Test set:     150000 samples

Class distribution in training set:
Status
Success    345632
Fail         4368
Name: count, dtype: int64

Class distribution in test set:
Status
Success    148128
Fail         1872
Name: count, dtype: int64

Failure rate (train): 1.25%
Failure rate (test):  1.25%


In [16]:
# =============================================================================
# 6.2 BASELINE MODEL (Predict Most Frequent Class)
# Reference: ML_Reference_Guide - Section 6.2
# =============================================================================

from sklearn.dummy import DummyClassifier

# Create baseline - always predict majority class
baseline = DummyClassifier(strategy='most_frequent')
baseline_results, baseline_pred = evaluate_classifier(
    baseline, X_train, X_test, y_train, y_test, 
    model_name="Baseline (Most Frequent)"
)
print_classification_report(baseline_results, y_test, baseline_pred)

MODEL: Baseline (Most Frequent)

Timing:
  Training Time:   0.0818 seconds
  Prediction Time: 0.0003 seconds

Overall Performance:
  Accuracy:        0.9875
  Macro F1:        0.4969

Minority Class (Fail) Performance:
  Precision:       0.0000
  Recall:          0.0000
  F1-Score:        0.0000

Majority Class (Success) Performance:
  Precision:       0.9875
  Recall:          1.0000
  F1-Score:        0.9937

Confusion Matrix:
              Predicted
              Success  Fail
  Actual Success  148128       0
  Actual Fail       1872       0
----------------------------------------------------------------------


In [17]:
# =============================================================================
# 6.3 LOGISTIC REGRESSION WITH CLASS WEIGHT BALANCING
# Reference: ML_Reference_Guide - Section 16 (SVM and Classification)
# =============================================================================

from sklearn.linear_model import LogisticRegression

# Logistic Regression with balanced class weights
lr_balanced = LogisticRegression(
    class_weight='balanced',  # Automatically adjusts weights inversely proportional to class frequencies
    max_iter=1000,
    random_state=42
)

lr_results, lr_pred = evaluate_classifier(
    lr_balanced, X_train, X_test, y_train, y_test,
    model_name="Logistic Regression (Balanced)"
)
print_classification_report(lr_results, y_test, lr_pred)

MODEL: Logistic Regression (Balanced)

Timing:
  Training Time:   1.3328 seconds
  Prediction Time: 0.0064 seconds

Overall Performance:
  Accuracy:        0.9214
  Macro F1:        0.5910

Minority Class (Fail) Performance:
  Precision:       0.1274
  Recall:          0.9060
  F1-Score:        0.2234

Majority Class (Success) Performance:
  Precision:       0.9987
  Recall:          0.9216
  F1-Score:        0.9586

Confusion Matrix:
              Predicted
              Success  Fail
  Actual Success  136511   11617
  Actual Fail        176    1696
----------------------------------------------------------------------


In [18]:
# =============================================================================
# 6.4 DECISION TREE WITH CLASS WEIGHT BALANCING
# Reference: ML_Reference_Guide - Section 14 (Decision Trees)
# =============================================================================

from sklearn.tree import DecisionTreeClassifier

# Decision Tree with balanced weights and depth limiting to prevent overfitting
dt_balanced = DecisionTreeClassifier(
    class_weight='balanced',
    max_depth=10,  # Limit depth to prevent overfitting
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)

dt_results, dt_pred = evaluate_classifier(
    dt_balanced, X_train, X_test, y_train, y_test,
    model_name="Decision Tree (Balanced, max_depth=10)"
)
print_classification_report(dt_results, y_test, dt_pred)

MODEL: Decision Tree (Balanced, max_depth=10)

Timing:
  Training Time:   0.4911 seconds
  Prediction Time: 0.0050 seconds

Overall Performance:
  Accuracy:        0.9214
  Macro F1:        0.5910

Minority Class (Fail) Performance:
  Precision:       0.1274
  Recall:          0.9060
  F1-Score:        0.2234

Majority Class (Success) Performance:
  Precision:       0.9987
  Recall:          0.9216
  F1-Score:        0.9586

Confusion Matrix:
              Predicted
              Success  Fail
  Actual Success  136511   11617
  Actual Fail        176    1696
----------------------------------------------------------------------


In [19]:
# =============================================================================
# 6.5 K-NEAREST NEIGHBORS
# Reference: ML_Reference_Guide - Section 16 (Comparing Models)
# Note: KNN does not have class_weight, so we scale features instead
# =============================================================================

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Scale features for KNN (distance-based algorithm)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# KNN with moderate k value
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)

knn_results, knn_pred = evaluate_classifier(
    knn, X_train_scaled, X_test_scaled, y_train, y_test,
    model_name="KNN (k=5, scaled features)"
)
print_classification_report(knn_results, y_test, knn_pred)

MODEL: KNN (k=5, scaled features)

Timing:
  Training Time:   0.1639 seconds
  Prediction Time: 10.2286 seconds

Overall Performance:
  Accuracy:        0.9879
  Macro F1:        0.5369

Minority Class (Fail) Performance:
  Precision:       0.7524
  Recall:          0.0422
  F1-Score:        0.0799

Majority Class (Success) Performance:
  Precision:       0.9880
  Recall:          0.9998
  F1-Score:        0.9939

Confusion Matrix:
              Predicted
              Success  Fail
  Actual Success  148102      26
  Actual Fail       1793      79
----------------------------------------------------------------------


In [20]:
# =============================================================================
# 6.6 SUPPORT VECTOR MACHINE WITH BALANCED WEIGHTS
# Reference: ML_Reference_Guide - Section 16 (SVM and Kernels)
# Note: Using subsample for SVM due to O(n²) complexity
# =============================================================================

from sklearn.svm import SVC

# SVM with RBF kernel and balanced weights
# Using a subsample due to computational complexity
sample_size = min(50000, len(X_train_scaled))
sample_idx = np.random.choice(len(X_train_scaled), sample_size, replace=False)
X_train_svm = X_train_scaled[sample_idx]
y_train_svm = y_train.iloc[sample_idx]

print(f"Training SVM on {sample_size} samples (subsample due to O(n²) complexity)...")

svm_balanced = SVC(
    kernel='rbf',
    class_weight='balanced',
    C=1.0,
    gamma='scale',
    random_state=42
)

svm_results, svm_pred = evaluate_classifier(
    svm_balanced, X_train_svm, X_test_scaled, y_train_svm, y_test,
    model_name="SVM RBF (Balanced, subsample=50k)"
)
print_classification_report(svm_results, y_test, svm_pred)

Training SVM on 50000 samples (subsample due to O(n²) complexity)...
MODEL: SVM RBF (Balanced, subsample=50k)

Timing:
  Training Time:   8.8979 seconds
  Prediction Time: 30.2759 seconds

Overall Performance:
  Accuracy:        0.9214
  Macro F1:        0.5909

Minority Class (Fail) Performance:
  Precision:       0.1273
  Recall:          0.9049
  F1-Score:        0.2232

Majority Class (Success) Performance:
  Precision:       0.9987
  Recall:          0.9216
  F1-Score:        0.9586

Confusion Matrix:
              Predicted
              Success  Fail
  Actual Success  136513   11615
  Actual Fail        178    1694
----------------------------------------------------------------------


In [21]:
# =============================================================================
# 7. MODEL COMPARISON SUMMARY
# Reference: ML_Reference_Guide - Section 8 (Model Evaluation and Selection)
# =============================================================================

# Collect all results
all_results = [baseline_results, lr_results, dt_results, knn_results, svm_results]

# Create summary DataFrame
summary_df = pd.DataFrame(all_results)
summary_df = summary_df[[
    'model_name', 'accuracy', 'f1_macro', 
    'precision_fail', 'recall_fail', 'f1_fail',
    'train_time', 'predict_time'
]]

# Format for display
summary_df['train_time'] = summary_df['train_time'].apply(lambda x: f"{x:.3f}s")
summary_df['predict_time'] = summary_df['predict_time'].apply(lambda x: f"{x:.3f}s")

# Round numeric columns
for col in ['accuracy', 'f1_macro', 'precision_fail', 'recall_fail', 'f1_fail']:
    summary_df[col] = summary_df[col].apply(lambda x: f"{x:.4f}")

print("=" * 100)
print("MODEL COMPARISON SUMMARY")
print("=" * 100)
print("\nFocus: Minority Class (Fail) Detection Performance")
print("-" * 100)
print(summary_df.to_string(index=False))
print("-" * 100)
print("\nKey Observations:")
print("- Accuracy alone is misleading due to class imbalance (baseline gets ~98.8% by predicting all Success)")
print("- F1-Fail measures balanced precision/recall for the minority class")
print("- High Recall-Fail means fewer missed authentication failures (false negatives)")
print("- High Precision-Fail means fewer false alarms (false positives)")
print("- Trade-off between interpretability (Decision Tree) and performance (SVM)")
print("=" * 100)

MODEL COMPARISON SUMMARY

Focus: Minority Class (Fail) Detection Performance
----------------------------------------------------------------------------------------------------
                            model_name accuracy f1_macro precision_fail recall_fail f1_fail train_time predict_time
              Baseline (Most Frequent)   0.9875   0.4969         0.0000      0.0000  0.0000     0.082s       0.000s
        Logistic Regression (Balanced)   0.9214   0.5910         0.1274      0.9060  0.2234     1.333s       0.006s
Decision Tree (Balanced, max_depth=10)   0.9214   0.5910         0.1274      0.9060  0.2234     0.491s       0.005s
            KNN (k=5, scaled features)   0.9879   0.5369         0.7524      0.0422  0.0799     0.164s      10.229s
     SVM RBF (Balanced, subsample=50k)   0.9214   0.5909         0.1273      0.9049  0.2232     8.898s      30.276s
----------------------------------------------------------------------------------------------------

Key Observations:
- Accu

In [ ]:
# =============================================================================
# 7.1 CONFUSION MATRIX PLOTS — ALL MODELS
# Visual comparison of model predictions
# =============================================================================

from sklearn.metrics import confusion_matrix

model_preds = {
    'Baseline (Most Frequent)': baseline_pred,
    'Logistic Regression': lr_pred,
    'Decision Tree': dt_pred,
    'KNN (k=5)': knn_pred,
    'SVM RBF': svm_pred
}

fig, axes = plt.subplots(1, 5, figsize=(24, 5))

for ax, (name, y_pred) in zip(axes, model_preds.items()):
    cm = confusion_matrix(y_test, y_pred, labels=['Success', 'Fail'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Success', 'Fail'],
                yticklabels=['Success', 'Fail'],
                ax=ax, cbar=False)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('Actual', fontsize=11)

plt.suptitle('Confusion Matrices — All Models', fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 7.2 MODEL PERFORMANCE BAR CHART — Minority Class Metrics
# =============================================================================

metrics_df = pd.DataFrame([lr_results, dt_results, knn_results, svm_results])

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(metrics_df))
width = 0.25

bars1 = ax.bar(x - width, metrics_df['precision_fail'], width, label='Precision (Fail)', 
               color='#3498db', edgecolor='black')
bars2 = ax.bar(x,         metrics_df['recall_fail'],    width, label='Recall (Fail)', 
               color='#e74c3c', edgecolor='black')
bars3 = ax.bar(x + width, metrics_df['f1_fail'],        width, label='F1 (Fail)', 
               color='#2ecc71', edgecolor='black')

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Minority Class (Fail) Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_df['model_name'], rotation=15, ha='right', fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random baseline')

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

---
## 8. Conclusions and What Classification Can Address

### What Classification CAN Address with This Data

1. **Behavioral Pattern Recognition**: Classification models can learn which combinations of AuthType, LogonType, and Activity are associated with authentication failures.

2. **Anomaly Detection Baseline**: By training on normal patterns, the models establish a baseline for what "normal" authentication looks like, enabling anomaly scoring.

3. **Feature Importance**: Decision Trees and Logistic Regression provide interpretable feature coefficients/importances showing which authentication behaviors are most predictive of failures.

4. **Real-time Scoring**: Fast models (Logistic Regression, Decision Tree) can score new authentication events in real-time for security monitoring.

### Limitations and Challenges

1. **Severe Class Imbalance**: With only ~1.2% failures, models tend to predict "Success" for everything. Class weighting helps but doesn't fully resolve this.

2. **Anonymized Timestamps**: The Time column has been classified by the laboratory, preventing temporal analysis (time-of-day patterns, event sequences, window aggregations).

3. **Identity vs Behavior**: High-cardinality identity features (users, computers) cannot be directly one-hot encoded. User-level aggregation provides an alternative.

4. **Context Dependency**: Authentication failures are often context-dependent (user history, sequence of events) - simple classification may miss these patterns.

5. **Evolving Patterns**: Attack patterns evolve; models need retraining as new authentication patterns emerge.

### Recommended Next Steps

1. **Expand User Profiles**: Create more sophisticated user-level behavioral aggregations (diversity metrics, typical behavior patterns)

2. **Ensemble Methods**: Random Forests or Gradient Boosting for improved performance on imbalanced data

3. **Anomaly Detection Framing**: Consider Isolation Forest or One-Class SVM to model "normal" behavior

4. **Cross-Validation**: Implement stratified k-fold CV for more robust performance estimates

5. **Threshold Tuning**: Optimize decision threshold based on business cost of false positives vs false negatives

---

## Summary

This analysis demonstrates a structured approach to classification on enterprise authentication data, following the CRISP-DM methodology and ML Reference Guide. Key findings:

- **Behavioral features** (AuthType, LogonType, Activity) can be effectively encoded using OneHotEncoding
- **Class imbalance** requires special handling (class_weight='balanced')
- **Standardized reporting** enables consistent model comparison
- **Trade-offs exist** between interpretability (Decision Trees) and performance (SVM)
- **Window-based features** provide a pathway for incorporating temporal and identity information without explosion in dimensionality